# Estrutura Principal - Transformer

In [4]:
import tiktoken

enc = tiktoken.get_encoding("gpt2")

texto = "Eu sou Gabriel, eu sou inteligente, eu sou eu"
token_ids = enc.encode(texto)

print(token_ids)
print(enc.n_vocab)  # vocab_size → 50257

[36, 84, 24049, 17371, 11, 304, 84, 24049, 33649, 328, 21872, 11, 304, 84, 24049, 304, 84]
50257


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class Transformer(nn.Module):
    def __init__(self, seq, d_model):
        super().__init__()
        self.seq = seq
        self.d_model = d_model

        pe = self.positional_encoding()
        self.register_buffer('pe', pe)

    def positional_encoding(self):

        pe = torch.zeros((self.seq, self.d_model))
        

        for pos in range(self.seq):
            for i in range(0, self.d_model, 2):
                pe[pos, i] = math.sin(pos / 10_000 ** (i/self.d_model))
                pe[pos, i+1] = math.cos(pos / 10_000 ** (i/self.d_model))

        return pe

In [2]:
class SelfAttention(nn.Module):
    def __init__(self, seq_len, d_model, causal=False):
        super().__init__()

        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.causal = causal
        self.register_buffer('mask', torch.tril(torch.ones(seq_len, seq_len)))

    def scaled_dot_product_attention(self, x):

        Q, K, V = self.W_Q(x), self.W_K(x), self.W_V(x)
        scores  = (Q @ K.transpose(-2,-1)) / math.sqrt(Q.shape[-1])

        

        if self.causal:

            masked_scores = scores.masked_fill(self.mask == 0, float('-inf'))
            weight = torch.softmax(masked_scores, dim=-1)

        else:

            weight = torch.softmax(scores, dim=-1)

        return weight @ V

In [3]:
model = Transformer(3, 512)
x = model.positional_encoding()

In [11]:
attention = SelfAttention(3, 512, True)
attention.scaled_dot_product_attention(x)

tensor([[-0.3168, -0.1223,  0.3862,  ...,  0.5271,  0.4207, -0.9127],
        [-0.3517, -0.0785,  0.3296,  ...,  0.5596,  0.3709, -0.8895],
        [-0.3778, -0.0329,  0.2866,  ...,  0.5911,  0.3139, -0.8671]],
       grad_fn=<MmBackward0>)

In [12]:
attention = SelfAttention(3, 512, False)
attention.scaled_dot_product_attention(x)

tensor([[ 0.1194,  0.0532,  0.0088,  ...,  0.6659, -0.3978,  0.3708],
        [ 0.1193,  0.0532,  0.0085,  ...,  0.6657, -0.3976,  0.3709],
        [ 0.1194,  0.0532,  0.0089,  ...,  0.6660, -0.3979,  0.3708]],
       grad_fn=<MmBackward0>)

In [6]:
class QKVAttention(nn.Module):
    def __init__(self, d_model, h, s, causal=False):
        super().__init__()

        self.h = h
        self.reduced_dimension = d_model // h
        self.seq = s
        self.d_model = d_model
        self.batch = 1 # Posso tornar isso dinâmico
        self.causal = causal

        if causal:
            self.register_buffer('mask', torch.tril(torch.ones(s, s)))
        else:
            self.mask = None

        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.W_O = nn.Linear(d_model, d_model, bias=False)

    def MultiHeadAttention(self, x):

        Q, K, V = self.W_Q(x), self.W_K(x), self.W_V(x)

        Q_i = Q.reshape(self.batch, self.seq, self.h, self.reduced_dimension).transpose(1, 2)
        K_i = K.reshape(self.batch, self.seq, self.h, self.reduced_dimension).transpose(1, 2)
        V_i = V.reshape(self.batch, self.seq, self.h, self.reduced_dimension).transpose(1, 2)

        score = (Q_i @ K_i.transpose(-2, -1)) / math.sqrt(self.reduced_dimension)

        if self.causal:        
            score = score.masked_fill(self.mask == 0, float('-inf'))
   
        weight = F.softmax(score, dim=-1) @ V_i
        weight = weight.transpose(1, 2).contiguous().view(self.batch, self.seq, self.d_model)


        return self.W_O(weight).squeeze(0)

In [7]:
QKV = QKVAttention(512, 8, 3, True)
QKV.MultiHeadAttention(x)

tensor([[ 0.1801, -0.3132, -0.1487,  ..., -0.3508, -0.0180, -0.0861],
        [ 0.2056, -0.2908, -0.1390,  ..., -0.3523,  0.0100, -0.1412],
        [ 0.2198, -0.2748, -0.1180,  ..., -0.3585,  0.0661, -0.1817]],
       grad_fn=<SqueezeBackward1>)